
# CSBS-BS Blind Scoring Clinician Assignment Algorithm

**Author:** Namit Shrivastava
**Context:** Assigning a "scoring clinician" to every CSBS-BS visit such that the assigned scorer (1) is never the visit's examiner, and (2) has the least prior familiarity with that child, prioritizing "blindness" to reduce recognition bias.

**Clinician pool:** \(C = \{\text{Emma}, \text{Tessa}, \text{Axie}\}\)



## 1. Notation and Definitions

Let a dataset consist of visit records \(r = (i, m, e)\), where:

- \(i\) = child ID
- \(m\) = visit month (age at visit, e.g. 9, 12, 15, 18, 24)
- \(e \in C\) = examiner for that visit (or NA if visit did not occur)

For a given child \(i\), let \(V(c, i)\) be the set of visit months where clinician \(c\) served as **examiner** for child \(i\):

\[
V(c, i) = \{\, m : (i, m, c) \in \text{Data}, \ e = c \,\}
\]

Let \(n(c, i) = |V(c, i)|\) be the number of times clinician \(c\) has examined child \(i\).

For a visit \(r=(i,m,e)\) needing a scorer, the candidate pool excludes the examiner:

\[
\text{Cand}(r) = C \setminus \{e\}
\]

Let \(W(c)\) denote the **running cumulative workload** — the total number of scoring assignments given to clinician \(c\) so far, updated sequentially as rows are processed.



## 2. Step 1 — Never Seen (NS)

A clinician who has **never** examined this child is the strongest candidate (guaranteed blind):

\[
\text{NS}(r) = \{\, c \in \text{Cand}(r) : n(c, i) = 0 \,\}
\]

**Decision rule:**

- If \(|\text{NS}(r)| = 1\): assign that clinician.
- If \(|\text{NS}(r)| = 2\) (tie, both never seen this child): assign the clinician with the **lower current workload**:
  \[
  c^* = \operatorname*{argmin}_{c \in \text{NS}(r)} W(c)
  \]
  If workloads are equal, break the tie **at random**.
- If \(\text{NS}(r) = \emptyset\): proceed to Step 2.



## 3. Step 2 — Least Visits (LV)

If both candidates have seen the child before, prefer whoever has seen them **least often**:

\[
m^{*} = \min_{c \in \text{Cand}(r)} n(c, i), \qquad
\text{LV}(r) = \{\, c \in \text{Cand}(r) : n(c, i) = m^{*} \,\}
\]

**Decision rule:**

- If \(|\text{LV}(r)| = 1\): assign that clinician.
- If \(|\text{LV}(r)| = 2\) (tie in visit counts): proceed to Step 3 (Furthest in Time), restricted to the tied clinicians.



## 4. Step 3 — Furthest in Time (FT)

Rationale (per project lead): a clinician is less likely to *recognize* a child from a visit that is far in age/time from any visit they personally conducted with that child. So for each tied candidate, we measure how close their **nearest own encounter** with the child is to the current visit month — and prefer the candidate whose nearest encounter is furthest away.

For each tied clinician \(c \in \text{LV}(r)\), define their distance to the current visit as the **minimum** absolute month-gap to any of their own visits with this child:

\[
d(c) = \min_{v \,\in\, V(c, i)} \; |m - v|
\]

**Decision rule:**

\[
c^{*} = \operatorname*{argmax}_{c \in \text{LV}(r)} d(c)
\]

- If there is still a tie in \(d(c)\): break by lower running workload \(W(c)\), then randomly if still tied.

## 5. NA Visits

If \(e = \text{NA}\) (visit did not occur), no scoring clinician is assigned — the row passes through with `Assigned Scoring Clinician = NA`, and it does **not** update anyone's \(n(c,i)\) or \(W(c)\).
